## API Call

In [1]:
import requests
import pandas as pd


def get_nsf_awards(first_name, middle_name, last_name, email, rpp=25):

    name_parts = [first_name, middle_name, last_name]

    name_parts = [
        str(x).strip()
        for x in name_parts
        if pd.notna(x)
        and str(x).strip() != ""
        and str(x).strip().lower() != "nan"
    ]

    person_name = " ".join(name_parts)

    print(
        f"Searching NSF: "
        f"{person_name}"
        )

    def search_awards(search_parameter):

        awards_all = []
        offset = 0

        while True:

            params = {
                search_parameter: person_name,
                "rpp": rpp,
                "offset": offset
            }

            response = requests.get(
                "https://api.nsf.gov/services/v1/awards.json",
                params=params,
                timeout=30
            )

            response.raise_for_status()

            awards = response.json()["response"].get("award", [])

            if not awards:
                break

            awards_all.extend(awards)

            if len(awards) < rpp:
                break

            offset += rpp

        return awards_all

    # Search PI and Co-PI awards
    pi_awards = search_awards("pdPIName")
    copi_awards = search_awards("coPDPI")

    # Add role
    for award in pi_awards:
        award["Role"] = "PI"

    for award in copi_awards:
        award["Role"] = "Co-PI"

    # Combine
    df = pd.DataFrame(pi_awards + copi_awards)

    # Add searched person's information
    df["First Name"] = first_name
    df["Middle Name"] = middle_name
    df["Last Name"] = last_name
    df["Searched Email"] = email

    # Desired output columns
    output_columns = [
        "First Name",
        "Middle Name",
        "Last Name",
        "Participants",
        "Searched Email",
        "Role",
        "Title",
        "Abstract",
        "Award Type",
        "Journal",
        "Amount",
        "NSF Email",
        "Institute",
        "Start Date",
        "End Date"
    ]

    # NSF field names
    columns = {
        "title": "Title",
        "abstractText": "Abstract",
        "transType": "Award Type",
        "jrnl": "Journal",
        "estimatedTotalAmt": "Amount",
        "piEmail": "NSF Email",
        "awardee": "Institute",
        "startDate": "Start Date",
        "expDate": "End Date"
    }

    # Rename fields that exist
    df = df.rename(columns=columns)

    # Make sure every desired column exists
    for col in output_columns:
        if col not in df.columns:
            df[col] = pd.NA

    # Return only desired columns, in the specified order
    df = df[output_columns]

    return df

In [2]:
df = pd.read_csv("scholar228_funding_triedby_junpeng.csv")[["identifier_first_name", "identifier_middle", "identifier_last_name", "identifier_email"]]

print(df)

    identifier_first_name identifier_middle identifier_last_name  \
0                Xiaojing               NaN                 Wang   
1                   Varun               NaN               Mishra   
2              Temiloluwa               NaN             Prioleau   
3                 Melissa               NaN                  Loh   
4          Yu Kuei (Alex)               NaN                  Lin   
..                    ...               ...                  ...   
222                   Bin               NaN                  Xie   
223               Mengjun               NaN                  Xie   
224                  Ting               NaN                  Zhu   
225                  Ingo               NaN               Vetter   
226                 Susan               NaN             Vorkoper   

            identifier_email  
0    xiaojing.wang@uconn.edu  
1      varun@varunmishra.com  
2    tprioleau@dartmouth.edu  
3       melissalkp@gmail.com  
4       yuklin@med.umich.edu

In [7]:
import time

def get_nsf_awards_for_people(people, delay=0.2):

    all_results = []

    for _, person in people.iterrows():

        first_name = person["identifier_first_name"]
        middle_name = person["identifier_middle"]
        last_name = person["identifier_last_name"]
        email = person["identifier_email"]

        try:

            result = get_nsf_awards(
                first_name=first_name,
                middle_name=middle_name,
                last_name=last_name,
                email=email
            )

            if not result.empty:
                all_results.append(result)

                print(
                    f"  Found {len(result)} matching award(s)"
                )

            else:
                print("  No matching awards found")

        except requests.exceptions.RequestException as e:

            print(
                f"  API error for "
                f"{first_name} {last_name}: {e}"
            )

        except Exception as e:

            print(
                f"  Error for "
                f"{first_name} {last_name}: {e}"
            )

        time.sleep(delay)

    if not all_results:
        return pd.DataFrame(
            columns=[
                "Title",
                "Abstract",
                "Award Type",
                "Amount",
                "Email",
                "Institute",
                "Date"
            ]
        )

    return pd.concat(
        all_results,
        ignore_index=True
    )

In [8]:
results = get_nsf_awards_for_people(df)

Searching NSF: Xiaojing Wang
  Found 2 matching award(s)
Searching NSF: Varun Mishra
  Found 1 matching award(s)
Searching NSF: Temiloluwa Prioleau
  Found 3 matching award(s)
Searching NSF: Melissa Loh
  No matching awards found
Searching NSF: Yu Kuei (Alex) Lin
  No matching awards found
Searching NSF: Shanthini Kasturi
  No matching awards found
Searching NSF: Carolyn Yeager
  No matching awards found
Searching NSF: Justin Knox
  No matching awards found
Searching NSF: Kara Christensen
  No matching awards found
Searching NSF: Laura Bellows
  No matching awards found
Searching NSF: Heather Nuske
  No matching awards found
Searching NSF: Caroline Silva
  No matching awards found
Searching NSF: Caroline Dorfman
  No matching awards found
Searching NSF: Mary Ellen Vajravelu
  No matching awards found
Searching NSF: Jessica Lipschitz
  No matching awards found
Searching NSF: Agatha Lenartowicz
  No matching awards found
Searching NSF: Jeff Epstein
  No matching awards found
Searching NS

In [9]:
results["Role"].unique()

array(['PI', 'Co-PI'], dtype=object)

In [10]:
results

,First Name,Middle Name,Last Name,Participants,Searched Email,Role,Title,Abstract,Award Type,Journal,Amount,NSF Email,Institute,Start Date,End Date
0,Xiaojing,NaN,Wang,NaN,xiaojing.wang@uconn.edu,PI,Conference: ICSA 2025 Applied Statistics Sympo...,This award will play a pivotal role in promoti...,Standard Grant,NaN,21000,xiaojing.wang@uconn.edu,UNIVERSITY OF CONNECTICUT,06/15/2025,05/31/2026
1,Xiaojing,NaN,Wang,NaN,xiaojing.wang@uconn.edu,PI,CAREER: Advancing Latent Variable Statistical ...,This research project will advance statistical...,Continuing Grant,[{'artTitl': 'A classification model for conti...,420000,xiaojing.wang@uconn.edu,UNIVERSITY OF CONNECTICUT,09/01/2019,08/31/2026
2,Varun,NaN,Mishra,NaN,varun@varunmishra.com,PI,CAREER: Context-aware framework to enable effe...,"Personal computing devices, such as smartphone...",Continuing Grant,[{'artTitl': 'Imputation Matters: A Deeper Loo...,679997,v.mishra@northeastern.edu,NORTHEASTERN UNIVERSITY,07/01/2025,06/30/2030
3,Temiloluwa,NaN,Prioleau,NaN,tprioleau@dartmouth.edu,PI,RAPID: Learning from the Experiences of COVID-...,Comprehensive knowledge about the novel corona...,Standard Grant,[{'artTitl': 'SymptomID: A Framework for Rapid...,199736,temiloluwa.o.prioleau@dartmouth.edu,TRUSTEES OF DARTMOUTH COLLEGE,07/01/2020,06/30/2022
4,Temiloluwa,NaN,Prioleau,NaN,tprioleau@dartmouth.edu,Co-PI,NRT: Transformative Research and Graduate Educ...,Sensors impact the lives of nearly every human...,Standard Grant,[{'artTitl': 'VERITAS Discovery of Very High E...,3000000,eric.r.fossum@dartmouth.edu,TRUSTEES OF DARTMOUTH COLLEGE,09/01/2021,08/31/2027
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
194,Ting,NaN,Zhu,NaN,NaN,Co-PI,Investigation of Degradation Mechanisms in Lay...,Non-Technical Abstract\r\nLithium ion batterie...,Continuing Grant,[{'artTitl': 'Avoiding fracture in a conversio...,460000,hailong.chen@me.gatech.edu,GEORGIA TECH RESEARCH CORP,09/01/2014,08/31/2018
195,Ting,NaN,Zhu,NaN,NaN,Co-PI,Processing Mechanically Reliable Flexible Orga...,Processing Mechanically Reliable Flexible Orga...,Standard Grant,"[{'artPageNum': '27169', 'artTitl': 'Environme...",420002,samuelg@umd.edu,GEORGIA TECH RESEARCH CORP,08/15/2014,06/30/2018
196,Ting,NaN,Zhu,NaN,NaN,Co-PI,Fabrication and Thermomechanical Characterizat...,Fabrication and Thermomechanical Characterizat...,Standard Grant,"[{'artPageNum': '883', 'artTitl': 'Patterning ...",350002,olivier.pierron@me.gatech.edu,GEORGIA TECH RESEARCH CORP,08/15/2008,07/31/2012
197,Ting,NaN,Zhu,NaN,NaN,Co-PI,SGER: Development of Thin Film Encapsulation L...,This Small Grant for Exploratory Research (SGE...,Standard Grant,"[{'artPageNum': '163308', 'artTitl': 'A Hybrid...",0,samuelg@umd.edu,GEORGIA TECH RESEARCH CORP,08/01/2008,07/31/2010


In [11]:
results.to_excel("NSF_raw.xlsx")

## Merge with Demographic Data

In [12]:
import pandas as pd

df_nsf = pd.read_excel("NSF_raw.xlsx")
df_demo = pd.read_csv("scholar176_with_demographic.csv")

In [13]:
df_nsf2 = df_nsf.copy()
df_demo2 = df_demo.copy()

df_demo2["name_clean"] = (
    df_demo2["name"]
    .astype("string")
    .str.strip()
    # Remove Dr. / Dr at the beginning
    .str.replace(r"^\s*Dr\.?\s+", "", regex=True, case=False)
    # Remove one-letter middle-name entries such as M or M.
    .str.replace(r"\s+[A-Za-z]\.?(?=\s+[A-Za-z])", "", regex=True)
    # Remove one-letter middle-name entries such as M or M., including at the beginning
    .str.replace(r"(?:^|\s+)[A-Za-z]\.?(?=\s+[A-Za-z])", "", regex=True)
    # Remove anything inside parentheses, including the parentheses
    .str.replace(r"\([^)]*\)", "", regex=True)
    # Remove everything after the first comma, including the comma
    .str.replace(r",.*$", "", regex=True)
    .str.strip()
)

# Split the cleaned name
name_parts = df_demo2["name_clean"].str.split()

df_demo2["_first_match"] = name_parts.str[0]
df_demo2["_last_match"] = name_parts.str[-1]


# --- Create matching keys for df_nsf ---

df_nsf2["_first_match"] = (
    df_nsf2["First Name"]
    .astype("string")
    .str.strip()
    .str.split()
    .str[0]
)

df_nsf2["_last_match"] = (
    df_nsf2["Last Name"]
    .astype("string")
    .str.strip()
    .str.split()
    .str[-1]
)


# --- Normalize capitalization for matching ---

df_demo2["_first_match"] = df_demo2["_first_match"].str.lower()
df_demo2["_last_match"] = df_demo2["_last_match"].str.lower()

df_nsf2["_first_match"] = df_nsf2["_first_match"].str.lower()
df_nsf2["_last_match"] = df_nsf2["_last_match"].str.lower()


# --- Information to add from df_demo ---

demo_columns = [
    "record_id",
    "name",
    "Cohort",
    "gender",
    "stage",
    "com_discipline",
    "region"
]

demo_info = df_demo2[
    ["_first_match", "_last_match"] + demo_columns
]


# --- Merge onto NSF data ---

df_nsf2 = df_nsf2.merge(
    demo_info,
    on=["_first_match", "_last_match"],
    how="left"
)


# --- Remove temporary matching columns ---

df_nsf2 = df_nsf2.drop(
    columns=["_first_match", "_last_match"]
)

In [14]:
print(df_demo2["name_clean"])

0            Xiaojing Wang
1             Varun Mishra
2      Temiloluwa Prioleau
3              Kah Poh Loh
4              Yu Kuei Lin
              ...         
171           Parya Saberi
172          Hillol Sarker
173         Johannes Thrul
174                Bin Xie
175            Mengjun Xie
Name: name_clean, Length: 176, dtype: string


In [15]:
df_nsf2

,Unnamed: 0,First Name,Middle Name,Last Name,Participants,Searched Email,Role,Title,Abstract,Award Type,...,Institute,Start Date,End Date,record_id,name,Cohort,gender,stage,com_discipline,region
0,0,Xiaojing,NaN,Wang,NaN,xiaojing.wang@uconn.edu,PI,Conference: ICSA 2025 Applied Statistics Sympo...,This award will play a pivotal role in promoti...,Standard Grant,...,UNIVERSITY OF CONNECTICUT,06/15/2025,05/31/2026,1.0,Xiaojing Wang,2022.0,male,mid-career,CS / Stats / Engineering / Data Science,Northeast
1,1,Xiaojing,NaN,Wang,NaN,xiaojing.wang@uconn.edu,PI,CAREER: Advancing Latent Variable Statistical ...,This research project will advance statistical...,Continuing Grant,...,UNIVERSITY OF CONNECTICUT,09/01/2019,08/31/2026,1.0,Xiaojing Wang,2022.0,male,mid-career,CS / Stats / Engineering / Data Science,Northeast
2,2,Varun,NaN,Mishra,NaN,varun@varunmishra.com,PI,CAREER: Context-aware framework to enable effe...,"Personal computing devices, such as smartphone...",Continuing Grant,...,NORTHEASTERN UNIVERSITY,07/01/2025,06/30/2030,2.0,Varun Mishra,2022.0,female,mid-career,CS / Stats / Engineering / Data Science,Northeast
3,3,Temiloluwa,NaN,Prioleau,NaN,tprioleau@dartmouth.edu,PI,RAPID: Learning from the Experiences of COVID-...,Comprehensive knowledge about the novel corona...,Standard Grant,...,TRUSTEES OF DARTMOUTH COLLEGE,07/01/2020,06/30/2022,3.0,Temiloluwa Prioleau,2022.0,male,mid-career,CS / Stats / Engineering / Data Science,Northeast
4,4,Temiloluwa,NaN,Prioleau,NaN,tprioleau@dartmouth.edu,Co-PI,NRT: Transformative Research and Graduate Educ...,Sensors impact the lives of nearly every human...,Standard Grant,...,TRUSTEES OF DARTMOUTH COLLEGE,09/01/2021,08/31/2027,3.0,Temiloluwa Prioleau,2022.0,male,mid-career,CS / Stats / Engineering / Data Science,Northeast
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
194,194,Ting,NaN,Zhu,NaN,NaN,Co-PI,Investigation of Degradation Mechanisms in Lay...,Non-Technical Abstract\r\nLithium ion batterie...,Continuing Grant,...,GEORGIA TECH RESEARCH CORP,09/01/2014,08/31/2018,NaN,NaN,NaN,NaN,NaN,NaN,NaN
195,195,Ting,NaN,Zhu,NaN,NaN,Co-PI,Processing Mechanically Reliable Flexible Orga...,Processing Mechanically Reliable Flexible Orga...,Standard Grant,...,GEORGIA TECH RESEARCH CORP,08/15/2014,06/30/2018,NaN,NaN,NaN,NaN,NaN,NaN,NaN
196,196,Ting,NaN,Zhu,NaN,NaN,Co-PI,Fabrication and Thermomechanical Characterizat...,Fabrication and Thermomechanical Characterizat...,Standard Grant,...,GEORGIA TECH RESEARCH CORP,08/15/2008,07/31/2012,NaN,NaN,NaN,NaN,NaN,NaN,NaN
197,197,Ting,NaN,Zhu,NaN,NaN,Co-PI,SGER: Development of Thin Film Encapsulation L...,This Small Grant for Exploratory Research (SGE...,Standard Grant,...,GEORGIA TECH RESEARCH CORP,08/01/2008,07/31/2010,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
df_nsf2.to_excel("NSF_demographic.xlsx", index=False)